# Video Process Augmentation Pipeline (02A)

Second-version dataset pipeline for training augmentation.

**Adds:**
1. Temporal crop augmentation
2. Horizontal flip augmentation
3. Gemini + rule-based caption augmentation
4. Training-ready manifests for augmented data

**Outputs** (in `data/processed_aug/`):
- `videos/` -- processed + augmented `.mp4` files
- `captions_augmented.json` -- augmented filename-to-caption mapping
- `videos.txt` -- one video path per line
- `prompts.txt` -- one caption per line (same order)


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from config.config import (
    CAPTION_AUG_RULE_VARIANTS,
    CAPTION_CONTEXT,
    DATASET_AUG_DIR,
    ENABLE_HORIZONTAL_FLIP,
    GEMINI_API_KEY,
    MAX_DURATION_SEC,
    MIN_DURATION_SEC,
    MIN_FRAMES,
    PER_VIDEO_CONTEXT,
    RAW_VIDEO_DIR,
    TARGET_FPS,
    TARGET_HEIGHT,
    TARGET_WIDTH,
    TEMPORAL_CROP_DURATION_SEC,
    TEMPORAL_CROPS_PER_VIDEO,
    TRIGGER_TOKEN,
    VIDEO_AUG_DIR,
)
from src.captioning.augment import caption_augmented_videos
from src.data.analyze import analyze_all
from src.data.augment import augment_processed_videos
from src.data.preview import preview_videos
from src.data.process import process_all
from src.training.validate import validate_dataset

Path(str(DATASET_AUG_DIR)).mkdir(parents=True, exist_ok=True)
Path(str(VIDEO_AUG_DIR)).mkdir(parents=True, exist_ok=True)

print("Augmentation Configuration")
print("=" * 60)
print(f"Raw video dir            : {RAW_VIDEO_DIR}")
print(f"Augmented dataset dir    : {DATASET_AUG_DIR}")
print(f"Augmented video dir      : {VIDEO_AUG_DIR}")
print(f"Target                   : {TARGET_WIDTH}x{TARGET_HEIGHT} @ {TARGET_FPS} fps")
print(f"Duration                 : {MIN_DURATION_SEC}-{MAX_DURATION_SEC} s")
print(f"Temporal crop duration   : {TEMPORAL_CROP_DURATION_SEC} s")
print(f"Temporal crops/video     : {TEMPORAL_CROPS_PER_VIDEO}")
print(f"Horizontal flip enabled  : {ENABLE_HORIZONTAL_FLIP}")
print(f"Caption rule variants    : {CAPTION_AUG_RULE_VARIANTS}")
print(f"Trigger token            : {TRIGGER_TOKEN}")


## Step 1: Analyze Raw Videos


In [ ]:
print("STEP 1: Analyzing raw videos")
print("=" * 60)

analysis = analyze_all(str(RAW_VIDEO_DIR), min_duration=MIN_DURATION_SEC)
if not analysis:
    raise RuntimeError(f"No valid videos found in {RAW_VIDEO_DIR}")

df_analysis = pd.DataFrame(analysis)
df_analysis["resolution"] = df_analysis["width"].astype(str) + "x" + df_analysis["height"].astype(str)

display_cols = ["filename", "resolution", "fps", "duration_sec", "mean_brightness", "is_likely_black"]
display(df_analysis[display_cols])


## Step 2: Process Base Videos


In [ ]:
print("STEP 2: Processing base videos")
print("=" * 60)

processed = process_all(
    video_analysis=analysis,
    output_dir=str(VIDEO_AUG_DIR),
    target_w=TARGET_WIDTH,
    target_h=TARGET_HEIGHT,
    target_fps=TARGET_FPS,
    max_duration=MAX_DURATION_SEC,
    min_duration=MIN_DURATION_SEC,
    min_frames=MIN_FRAMES,
)

if not processed:
    raise RuntimeError("No videos survived base processing.")

df_processed = pd.DataFrame([
    {
        "original": v["original"],
        "processed": v["processed"],
        "resolution": f"{v['info']['width']}x{v['info']['height']}",
        "fps": v["info"]["fps"],
        "frames": v["info"]["frame_count"],
        "duration_sec": v["info"]["duration_sec"],
    }
    for v in processed
])
display(df_processed)


## Step 3: Augment Videos (Temporal Crops + Flips)


In [ ]:
print("STEP 3: Creating augmentations")
print("=" * 60)

augmented_records = augment_processed_videos(
    processed_videos=processed,
    output_dir=str(VIDEO_AUG_DIR),
    temporal_crop_duration_sec=TEMPORAL_CROP_DURATION_SEC,
    temporal_crops_per_video=TEMPORAL_CROPS_PER_VIDEO,
    include_horizontal_flip=ENABLE_HORIZONTAL_FLIP,
)

if not augmented_records:
    raise RuntimeError("Augmentation did not produce any records.")

df_aug = pd.DataFrame(augmented_records)
print(f"Total augmented dataset records: {len(df_aug)}")
display(df_aug[["filename", "source", "augmentation", "is_flipped", "temporal_start_sec"]].head(30))


## Step 4: Preview Augmented Videos


In [ ]:
%matplotlib inline

print("STEP 4: Preview augmented videos")
print("=" * 60)

preview_records = []
for rec in augmented_records[: min(8, len(augmented_records))]:
    preview_records.append({
        "path": rec["path"],
        "original": f"{rec['source']} | {rec['augmentation']}",
    })

preview_videos(preview_records, TARGET_WIDTH, TARGET_HEIGHT, TARGET_FPS)


## Step 5: Caption Augmented Videos (Gemini + Rules)


In [ ]:
print("STEP 5: Captioning augmented videos")
print("=" * 60)

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set. Add it to your .env file at the project root.")

captions_aug = caption_augmented_videos(
    video_records=augmented_records,
    video_dir=str(VIDEO_AUG_DIR),
    dataset_dir=str(DATASET_AUG_DIR),
    trigger_token=TRIGGER_TOKEN,
    context=CAPTION_CONTEXT,
    api_key=GEMINI_API_KEY,
    per_video_context=PER_VIDEO_CONTEXT,
    delay=1.0,
    rule_variants=CAPTION_AUG_RULE_VARIANTS,
)

df_captions = pd.DataFrame([(fn, cap) for fn, cap in captions_aug.items()], columns=["filename", "caption"])
df_captions["status"] = df_captions["caption"].apply(lambda c: "FAILED" if c.startswith("[FAILED]") else "OK")

total = len(df_captions)
success = (df_captions["status"] == "OK").sum()
failed = total - success

print(f"
Total: {total}  |  Success: {success}  |  Failed: {failed}")
if failed > 0:
    print("
FAILED captions:")
    display(df_captions[df_captions["status"] == "FAILED"])

display(df_captions[["filename", "caption"]].head(20))


## Step 6: Validate Augmented Dataset Manifests


In [ ]:
print("STEP 6: Validating augmented dataset")
print("=" * 60)

is_valid = validate_dataset(
    dataset_dir=str(DATASET_AUG_DIR),
    target_w=TARGET_WIDTH,
    target_h=TARGET_HEIGHT,
    min_frames=MIN_FRAMES,
)

if not is_valid:
    raise RuntimeError("Augmented dataset has errors. Fix them before training.")

videos_txt = os.path.join(str(DATASET_AUG_DIR), "videos.txt")
prompts_txt = os.path.join(str(DATASET_AUG_DIR), "prompts.txt")

with open(videos_txt) as f:
    n_videos = len([line for line in f.read().strip().splitlines() if line])
with open(prompts_txt) as f:
    n_prompts = len([line for line in f.read().strip().splitlines() if line])

assert n_videos == n_prompts, (
    f"Line count mismatch: videos.txt has {n_videos}, prompts.txt has {n_prompts}"
)

print(f"
videos.txt:  {n_videos} entries")
print(f"prompts.txt: {n_prompts} entries")
print(f"
Output directory: {DATASET_AUG_DIR}")
print("
Augmented dataset is ready for training.")
